In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path.cwd()
ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "05_tcga_classification_analysis"
if not ANALYSIS_DIR.exists():
    ANALYSIS_DIR = PROJECT_ROOT
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from utils.tcga_ml_pipeline import (
    RANDOM_SEED,
    build_training_observed_cpg_position_lookup,
    collect_all_pmds,
    filter_regions_by_training_cpg_coverage,
    fit_gamma_length_distribution,
    get_training_pmds,
    load_cgis,
    load_methylation_data,
    load_pmds_per_sample,
    load_tcga_samples,
    make_patient_groups,
    make_sample_types,
    pick_features,
    plot_classification_box_results,
    plot_classification_results,
    plot_sampled_feature_length_distributions,
    run_feature_classification,
)


In [ ]:
FEATURE_COUNTS = [1, 3, 5, 10, 50, 100]
N_SPLITS = 5
N_REPEATS = 10
N_FEATURE_DRAWS = 10
MIN_TRAINING_OBSERVED_CPGS_PER_REGION = 1
MIN_CPGS_FOR_RANDOM_REGIONS = 1


In [ ]:
samples_info = load_tcga_samples("TCGA-BRCA").sort_values("sample").reset_index(drop=True)
meth_data = load_methylation_data(samples_info)
sample_types = make_sample_types(samples_info)
sample_groups = make_patient_groups(samples_info)
pmds_per_sample = load_pmds_per_sample(samples_info)
cgis = load_cgis()

all_tumor_pmds = get_training_pmds(
    samples_info["sample"].tolist(),
    sample_types,
    pmds_per_sample,
    label="Tumor",
)
training_observed_cpgs = build_training_observed_cpg_position_lookup(
    meth_data,
    samples_info["sample"].tolist(),
)
eligible_tumor_pmds = filter_regions_by_training_cpg_coverage(
    all_tumor_pmds,
    training_observed_cpgs,
    min_cpgs=MIN_TRAINING_OBSERVED_CPGS_PER_REGION,
)
eligible_cgis = filter_regions_by_training_cpg_coverage(
    cgis,
    training_observed_cpgs,
    min_cpgs=MIN_TRAINING_OBSERVED_CPGS_PER_REGION,
)
features = pick_features(
    5,
    eligible_tumor_pmds,
    eligible_cgis,
    offset=0,
    random_seed=RANDOM_SEED,
    cpg_positions_by_chrom=training_observed_cpgs,
    min_cpgs_for_random_regions=MIN_CPGS_FOR_RANDOM_REGIONS,
    control_excluded_regions=all_tumor_pmds,
)


In [ ]:
{name: feature_df.head() for name, feature_df in features.items()}


In [ ]:
sns.set_theme(style="whitegrid")

pmd_length_distribution = fit_gamma_length_distribution(eligible_tumor_pmds)
cgi_length_distribution = fit_gamma_length_distribution(eligible_cgis)

pmd_lengths = eligible_tumor_pmds["length"].to_numpy(dtype=float)
cgi_lengths = eligible_cgis["length"].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pmd_x = np.linspace(pmd_lengths.min(), pmd_lengths.max(), 500)
axes[0].hist(pmd_lengths, bins=40, density=True, alpha=0.35, color="#1f77b4", label="Observed PMD lengths")
axes[0].plot(
    pmd_x,
    stats.gamma.pdf(
        pmd_x,
        a=pmd_length_distribution["shape"],
        loc=pmd_length_distribution["loc"],
        scale=pmd_length_distribution["scale"],
    ),
    color="#0b5394",
    linewidth=2,
    label="Gamma fit",
)
axes[0].set_title("PMD length distribution")
axes[0].set_xlabel("Length (bp)")
axes[0].set_ylabel("Density")
axes[0].legend(frameon=False)

cgi_x = np.linspace(cgi_lengths.min(), cgi_lengths.max(), 500)
axes[1].hist(cgi_lengths, bins=40, density=True, alpha=0.35, color="#2ca02c", label="Observed CGI lengths")
axes[1].plot(
    cgi_x,
    stats.gamma.pdf(
        cgi_x,
        a=cgi_length_distribution["shape"],
        loc=cgi_length_distribution["loc"],
        scale=cgi_length_distribution["scale"],
    ),
    color="#1b7f3b",
    linewidth=2,
    label="Gamma fit",
)
axes[1].set_title("CGI length distribution")
axes[1].set_xlabel("Length (bp)")
axes[1].set_ylabel("Density")
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()

feature_length_df = pd.concat(
    [
        pd.DataFrame({"feature_set": name, "length": feature_df["length"].to_numpy(dtype=float)})
        for name, feature_df in features.items()
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(12, 6))
feature_palette = {
    "PMD": "#0b5394",
    "Random PMD": "#3d85c6",
    "CGI": "#38761d",
    "Random long": "#9c6ade",
    "Random short": "#e69138",
}

for feature_set, feature_df in feature_length_df.groupby("feature_set", sort=False):
    lengths = feature_df["length"].dropna().astype(float)
    if lengths.empty:
        continue

    color = feature_palette.get(feature_set)
    label = f"{feature_set} (n={len(lengths)})"
    if lengths.nunique() > 1:
        sns.kdeplot(
            x=lengths,
            ax=ax,
            log_scale=True,
            common_norm=False,
            bw_adjust=0.9,
            fill=False,
            linewidth=2,
            color=color,
            label=label,
        )
    else:
        ax.axvline(lengths.iloc[0], color=color, linewidth=2, linestyle="--", label=label)

ax.set_title("Length distributions for sampled PMD, CGI, and random feature sets")
ax.set_xlabel("Length (bp, log scale)")
ax.set_ylabel("Density")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
samples_info

In [ ]:
sample_types = make_sample_types(samples_info)
sample_groups = make_patient_groups(samples_info)
pd.Series(sample_types).value_counts().rename_axis("sample_label").to_frame("n_samples")


In [ ]:
results, sampled_feature_regions = run_feature_classification(
    meth_data=meth_data,
    sample_types=sample_types,
    sample_groups=sample_groups,
    pmds_per_sample=pmds_per_sample,
    cgis=cgis,
    feature_counts=FEATURE_COUNTS,
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    n_feature_draws=N_FEATURE_DRAWS,
    random_seed=RANDOM_SEED,
    exclude_top_normal_shared_pmds=True,
    min_cpgs_for_random_regions=MIN_CPGS_FOR_RANDOM_REGIONS,
    min_training_observed_cpgs_per_region=MIN_TRAINING_OBSERVED_CPGS_PER_REGION,
)


In [ ]:
X_LOG_SCALE = False

classification_summary = plot_classification_results(results, x_log_scale=X_LOG_SCALE)
classification_box_data = plot_classification_box_results(results, n_features=5)
classification_length_data = plot_sampled_feature_length_distributions(
    sampled_feature_regions,
    n_features=None,
)
classification_summary.head(10)
